# Phi-4-mini-instruct SFT 파인튜닝
담당: Generation 파트 (한의정)

- 모델: microsoft/Phi-4-mini-instruct
- 환경: Colab H100 / A100 (bfloat16 풀정밀도)
- 데이터: kh_v3.json (입찰 공고 청크)
- 포맷: Phi-4-mini 채팅 포맷 (`<|system|>`, `<|user|>`, `<|assistant|>`)

In [7]:
!pip install -q --upgrade transformers peft trl accelerate datasets sentencepiece protobuf "torchao>=0.16.0"
!rm -rf ~/.cache/huggingface/modules/

import importlib
for pkg in ["transformers", "peft", "trl", "accelerate", "torchao"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<15} : {m.__version__}")
    except Exception as e:
        print(f"{pkg:<15} : ERROR - {e}")

print("\n설치 완료. 런타임 재시작 후 섹션 2부터 실행하세요.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.4 MB/s eta 0:00:00
transformers    : 5.9.0
peft            : 0.19.1


trl             : 1.5.1
accelerate      : 1.13.0
torchao         : 0.17.0

설치 완료. 런타임 재시작 후 섹션 2부터 실행하세요.


In [11]:
!rm -rf ~/.cache/huggingface/modules/transformers_modules/microsoft/
!pip install -q "transformers>=4.51.0"
import transformers
print(transformers.__version__)

In [1]:
# ── 섹션 1. GPU / Drive 마운트 확인 ──────────────────────────────────
import torch

assert torch.cuda.is_available(), "GPU가 필요합니다."
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {total_vram:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ── 섹션 2. 경로 및 하이퍼파라미터 ───────────────────────────────────
import os

CHUNK_JSON_PATH = "/content/drive/MyDrive/data/bidmate/kh_v3.json"
EVAL_CSV_DIR    = "/content/drive/MyDrive/data/bidmate/eval"  # 없으면 None으로
OUTPUT_DIR      = "/content/drive/MyDrive/data/bidmate/peft_output/phi4-mini"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "microsoft/Phi-4-mini-instruct"

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# 학습 — H100 기준 최적화
MAX_SEQ_LEN   = 2048
BATCH_SIZE    = 8      # H100 80GB → 크게 잡음
GRAD_ACCUM    = 2      # effective batch = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.05
SAVE_STEPS    = 100

# 데이터
CHUNK_SAMPLE_N    = 5000
CHUNK_MIN_LEN     = 100
TRAIN_SPLIT_RATIO = 0.9

print(f"모델  : {MODEL_ID}")
print(f"출력  : {OUTPUT_DIR}")
print(f"VRAM  : {total_vram:.1f}GB | batch={BATCH_SIZE} | grad_accum={GRAD_ACCUM}")

모델  : microsoft/Phi-4-mini-instruct
출력  : /content/drive/MyDrive/data/bidmate/peft_output/phi4-mini
VRAM  : 95.0GB | batch=8 | grad_accum=2


In [3]:
# ── 섹션 3. 데이터 로드 및 전처리 ────────────────────────────────────
import json, glob, ast, random
import pandas as pd
from collections import defaultdict
from datasets import Dataset

random.seed(42)

# 청크 로드
with open(CHUNK_JSON_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
print(f"총 청크 수: {len(chunks):,}")

chunk_by_src = defaultdict(list)
for c in chunks:
    src = c.get("metadata", {}).get("source_file", "")
    chunk_by_src[src].append(c)

# eval CSV 로드 (없으면 스킵)
eval_df = None
if EVAL_CSV_DIR and os.path.exists(EVAL_CSV_DIR):
    def safe_parse(val):
        if pd.isna(val) or str(val).strip() in ("", "[]", "{}", "nan"):
            return []
        try:
            return json.loads(val)
        except Exception:
            try:
                return ast.literal_eval(val)
            except Exception:
                return val

    csv_files = sorted(glob.glob(os.path.join(EVAL_CSV_DIR, "*.csv")))
    if csv_files:
        eval_df = pd.concat([pd.read_csv(p, dtype=str) for p in csv_files], ignore_index=True)
        for col in ["ground_truth_docs", "metadata_filter", "history"]:
            if col in eval_df.columns:
                eval_df[col] = eval_df[col].apply(safe_parse)
        print(f"eval CSV {len(csv_files)}개 | 레코드 {len(eval_df):,}개")
        print(eval_df["type"].value_counts().sort_index().to_string())
    else:
        print("eval CSV 없음 → 스킵")
else:
    print("eval CSV 경로 없음 → 스킵")

총 청크 수: 38,287
eval CSV 38개 | 레코드 1,100개
type
A    328
B    311
C    143
D    162
E    156


In [4]:
# ── 섹션 4. Phi-4-mini 포맷으로 데이터셋 구성 ────────────────────────
# Phi-4-mini 채팅 포맷:
#   <|system|>...<|end|>
#   <|user|>...<|end|>
#   <|assistant|>...<|end|>

SYSTEM_PROMPT = (
    "당신은 공공기관 및 기업의 제안요청서(RFP) 분석 전문가입니다. "
    "주어진 문서 내용을 바탕으로 정확하고 간결하게 답변하세요. "
    "문서에 없는 정보는 추측하지 말고 '해당 정보는 문서에서 확인되지 않습니다'라고 답하세요."
)

def is_valid_chunk(text: str) -> bool:
    return len(text.replace("[TABLE]", "").replace("[IMAGE]", "").strip()) >= CHUNK_MIN_LEN

def retrieve_context(ground_truth_docs: list, max_chunks: int = 3) -> str:
    contexts = []
    for fname in ground_truth_docs:
        stem = fname.replace(".hwp", "").replace(".pdf", "").strip()
        for src, doc_chunks in chunk_by_src.items():
            if stem in src.replace(".hwp", "").replace(".pdf", "").strip():
                for c in doc_chunks[:max_chunks]:
                    cleaned = c["text"].replace("[TABLE]", "").replace("[IMAGE]", "").strip()
                    if cleaned:
                        contexts.append(cleaned)
                break
    return "\n\n".join(contexts[:max_chunks])

def fmt_phi(s: dict) -> dict:
    """Phi-4-mini 채팅 포맷으로 변환"""
    return {"text": (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{s['instruction']}<|end|>\n"
        f"<|assistant|>\n{s['response']}<|end|>"
    )}

def make_dataset(raw: list) -> Dataset:
    return Dataset.from_list([fmt_phi(s) for s in raw])

# 청크 기반 요약/QA 샘플 생성 (Gemma 코드와 동일 방식)
valid_chunks = [c for c in chunks if is_valid_chunk(c["text"])]
sampled      = random.sample(valid_chunks, min(CHUNK_SAMPLE_N, len(valid_chunks)))

summary_samples = []
for c in sampled:
    text = c["text"].replace("[TABLE]", "").replace("[IMAGE]", "").strip()
    meta = c.get("metadata", {})
    agency  = meta.get("agency", "공공기관")
    project = meta.get("project_name", "정보화 사업")

    instruction = (
        f"[참고 문서]\n{text}\n\n"
        f"[질문]\n{agency}의 '{project}' 사업 내용을 요약하고 핵심 요건을 설명하세요."
    )
    response = (
        f"{agency}의 '{project}' 사업은 다음과 같은 내용을 포함합니다:\n"
        f"{text[:300]}..."
    ) if len(text) > 300 else (
        f"{agency}의 '{project}' 사업 내용: {text}"
    )
    summary_samples.append({"instruction": instruction, "response": response})

# eval CSV 기반 QA 샘플 추가 (있는 경우)
qa_samples = []
if eval_df is not None:
    for _, row in eval_df.iterrows():
        gt_docs  = row.get("ground_truth_docs", [])
        gt_ans   = str(row.get("ground_truth_answer", "")).strip()
        question = str(row.get("question", "")).strip()
        if not question or not gt_ans or gt_ans in ("nan", ""):
            continue
        context = retrieve_context(gt_docs if isinstance(gt_docs, list) else [], max_chunks=3)
        instruction = f"[참고 문서]\n{context}\n\n[질문]\n{question}" if context else question
        qa_samples.append({"instruction": instruction, "response": gt_ans})

all_samples = summary_samples + qa_samples
random.shuffle(all_samples)

split_idx   = int(len(all_samples) * TRAIN_SPLIT_RATIO)
train_raw   = all_samples[:split_idx]
eval_raw    = all_samples[split_idx:]

train_dataset = make_dataset(train_raw)
eval_dataset  = make_dataset(eval_raw)

print(f"학습 샘플: {len(train_dataset):,} | 평가 샘플: {len(eval_dataset):,}")
print("\n샘플 예시:")
print(train_dataset[0]["text"][:300])

학습 샘플: 5,490 | 평가 샘플: 610

샘플 예시:
<|system|>
당신은 공공기관 및 기업의 제안요청서(RFP) 분석 전문가입니다. 주어진 문서 내용을 바탕으로 정확하고 간결하게 답변하세요. 문서에 없는 정보는 추측하지 말고 '해당 정보는 문서에서 확인되지 않습니다'라고 답하세요.<|end|>
<|user|>
[참고 문서]
1. 기술평가기준
[기술제안서 평가항목 및 배점한도]



2. 가격평가기준
- 입찰가격이 추정가격의 100분의 80 이상인 경우



- 입찰가격이 추정가격의 100분의 80 미만인 경우
ㆍ평점 = 입찰가격이 추정가격의 100분의 80일 경우의 평점






In [5]:
# ── 섹션 5. 모델 로드 (bfloat16 풀정밀도, H100 최적화) ───────────────
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

total_vram  = torch.cuda.get_device_properties(0).total_memory / 1024**3
max_gpu_mem = f"{int(total_vram - 4)}GiB"
print(f"VRAM: {total_vram:.1f}GB | max_memory: {max_gpu_mem}")

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LEN

# 모델 — bfloat16 풀정밀도 (H100/A100 권장)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: max_gpu_mem, "cpu": "10GiB"},
)
model.config.use_cache = False

# LoRA 설정
lora_config = LoraConfig(
    r             = LORA_R,
    lora_alpha    = LORA_ALPHA,
    lora_dropout  = LORA_DROPOUT,
    target_modules= LORA_TARGET_MODULES,
    bias          = "none",
    task_type     = "CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n모델 로드 완료: {MODEL_ID}")

VRAM: 95.0GB | max_memory: 90GiB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

trainable params: 8,912,896 || all params: 3,844,934,656 || trainable%: 0.2318

모델 로드 완료: microsoft/Phi-4-mini-instruct


In [8]:
# ── 섹션 6. 학습 ─────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from trl import SFTTrainer, SFTConfig

total_steps  = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
print(f"전체 스텝: {total_steps} | warmup: {warmup_steps}")

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_steps                = warmup_steps,
    bf16                        = True,
    fp16                        = False,
    tf32                        = True,
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = SAVE_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_num_workers      = 2,
)

tokenizer.model_max_length = MAX_SEQ_LEN

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = eval_dataset,
    processing_class = tokenizer,
)

trainer.train()
print("\n학습 완료")

전체 스텝: 1029 | warmup: 51


Adding EOS to train dataset:   0%|          | 0/5490 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5490 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/610 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/610 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.518626,1.486309,1.594904,1050068.000000,0.704153
200,1.290826,1.315107,1.451806,2111006.000000,0.729765
300,1.132760,1.201747,1.327263,3171766.000000,0.749237
400,1.091016,1.105959,1.218465,4220745.000000,0.768018
500,0.949920,1.024723,1.138694,5264220.000000,0.784362
600,0.918876,0.965639,1.074140,6341392.000000,0.797551
700,0.859556,0.923122,1.021616,7383872.000000,0.807297
800,0.823770,0.895120,1.013581,8440982.000000,0.813359
900,0.808057,0.881034,0.998203,9494451.000000,0.816737
1000,0.799417,0.877539,0.997181,10557981.000000,0.817229



학습 완료


In [9]:
# ── 섹션 7. Loss Curve 저장 ───────────────────────────────────────────
log = trainer.state.log_history

train_steps  = [x["step"]      for x in log if "loss"      in x]
train_losses = [x["loss"]      for x in log if "loss"      in x]
eval_steps   = [x["step"]      for x in log if "eval_loss" in x]
eval_losses  = [x["eval_loss"] for x in log if "eval_loss" in x]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="train loss", alpha=0.7)
if eval_losses:
    plt.plot(eval_steps, eval_losses, label="eval loss", marker="o")
plt.title("Phi-4-mini Loss Curve")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=150)
plt.show()

best = min(eval_losses) if eval_losses else None
print(f"최저 eval loss: {best:.4f}" if best else "eval loss 없음")

최저 eval loss: 0.8774


In [10]:
# ── 섹션 8. LoRA 어댑터 저장 ─────────────────────────────────────────
adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"어댑터 저장 완료: {adapter_path}")

# GCP 배포 시 환경변수 설정:
# export PHI_ADAPTER_PATH=/home/euijeong/.../phi4-mini/lora_adapter

어댑터 저장 완료: /content/drive/MyDrive/data/bidmate/peft_output/phi4-mini/lora_adapter


In [11]:
# ── 섹션 9. 추론 테스트 ──────────────────────────────────────────────
import torch

model.eval()

test_questions = [
    "한국가스공사의 차세대 ERP 구축 사업 예산은 얼마입니까?",
    "고려대학교의 차세대 포털 학사 정보시스템 구축사업의 핵심 목표는 무엇입니까?",
]

for q in test_questions:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{q}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = 256,
            do_sample         = False,
            pad_token_id      = tokenizer.pad_token_id,
            eos_token_id      = tokenizer.convert_tokens_to_ids("<|end|>"),
        )
    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    print(f"\nQ: {q}")
    print(f"A: {pred}")
    print("-" * 60)


Q: 한국가스공사의 차세대 ERP 구축 사업 예산은 얼마입니까?
A: 이 사업의 예산은 250,000,000원(￦￦Two Hundred Fifty Million 원)입니다.
------------------------------------------------------------

Q: 고려대학교의 차세대 포털 학사 정보시스템 구축사업의 핵심 목표는 무엇입니까?
A: 고려대학교의 차세대 포털 학사 정보시스템 구축사업의 핵심 목표는 '학사 정보시스템의 기능 개선 및 시스템의 안정적 운영을 위한 지속적인 유지보수 및 개선'입니다.
------------------------------------------------------------
